In [1]:
import os
import json
import numpy as np
import pandas as pd

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Paths
INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Locate the raw file regardless of exact dataset slug/folder naming
raw_path = None
for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f.lower() == "creditcard.csv":
            raw_path = os.path.join(root, f)
            break
    if raw_path:
        break

if raw_path is None:
    raise FileNotFoundError(
        "creditcard.csv not found under /kaggle/input. "
        "Add the 'Credit Card Fraud Detection' dataset (mlg-ulb/creditcardfraud) via + Add Data."
    )

print(f"Found raw file at: {raw_path}")

# Config dict we'll keep building throughout NB0a and save at the end
config = {
    "notebook": "NB0a",
    "dataset": "creditcardfraud",
    "source": "mlg-ulb/creditcardfraud (Kaggle)",
    "seed": SEED,
    "raw_path": raw_path,
}

Found raw file at: /kaggle/input/datasets/mlg-ulb/creditcardfraud/creditcard.csv


In [2]:
df = pd.read_csv(raw_path)

print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nNulls per column:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())

print("\nOverall class balance:")
print(df["Class"].value_counts())
print(df["Class"].value_counts(normalize=True))

print("\nTime column range (seconds):", df["Time"].min(), "to", df["Time"].max())
print("Time range in hours:", df["Time"].max() / 3600)

# Preview the day-based temporal cutoff at the 24h boundary
DAY_CUTOFF_SECONDS = 86400
day1 = df[df["Time"] < DAY_CUTOFF_SECONDS]
day2 = df[df["Time"] >= DAY_CUTOFF_SECONDS]

print(f"\nDay 1 (Time < {DAY_CUTOFF_SECONDS}): {len(day1)} rows")
print(day1["Class"].value_counts())
print(f"\nDay 2 (Time >= {DAY_CUTOFF_SECONDS}): {len(day2)} rows")
print(day2["Class"].value_counts())

df.head()

Shape: (284807, 31)

Dtypes:
 Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

Nulls per column:
 Series([], dtype: int64)

Duplicate rows: 1081

Overall class balance:
Class
0    284315
1       492
Name: count, dtype: int64
Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

Time column range (seconds): 0.0 to 172792.0
Time range in hours: 47.99777777777778

Day 1 (Time < 86400): 144786 rows
Class
0    144505
1       281
Name: count, dtype: int64

Day 2 (

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
# Drop exact duplicate rows (known artifact of this dataset)
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_after = len(df)
print(f"Dropped {n_before - n_after} duplicate rows ({n_before} -> {n_after})")

DAY_CUTOFF_SECONDS = 86400
day1 = df[df["Time"] < DAY_CUTOFF_SECONDS].copy()
day2 = df[df["Time"] >= DAY_CUTOFF_SECONDS].copy()

day1_benign = day1[day1["Class"] == 0].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
day1_fraud  = day1[day1["Class"] == 1].sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Day 1 benign: 70 / 15 / 15
n_b = len(day1_benign)
n_train_end = int(0.70 * n_b)
n_id_end = int(0.85 * n_b)

train_df = day1_benign.iloc[:n_train_end].copy()
id_benign = day1_benign.iloc[n_train_end:n_id_end].copy()
id2_benign = day1_benign.iloc[n_id_end:].copy()

# Day 1 fraud: 50 / 50, none into train
n_f = len(day1_fraud)
n_fraud_half = n_f // 2
id_fraud = day1_fraud.iloc[:n_fraud_half].copy()
id2_fraud = day1_fraud.iloc[n_fraud_half:].copy()

id_df = pd.concat([id_benign, id_fraud], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
id2_df = pd.concat([id2_benign, id2_fraud], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
ood_df = day2.reset_index(drop=True)

# --- Verification ---
print("\nSplit sizes and class balance:")
for name, d in [("Train", train_df), ("ID", id_df), ("ID2", id2_df), ("OOD", ood_df)]:
    print(f"\n{name}: {len(d)} rows")
    print(d["Class"].value_counts().to_dict())

# Sanity check: no index overlap between Train/ID/ID2 (using Time+Amount as a row fingerprint
# since we reset_index — safer to check via a merge on all original columns)
key_cols = [c for c in df.columns if c != "Class"]
train_keys = set(map(tuple, train_df[key_cols].values))
id_keys = set(map(tuple, id_df[key_cols].values))
id2_keys = set(map(tuple, id2_df[key_cols].values))

print("\nOverlap Train/ID:", len(train_keys & id_keys))
print("Overlap Train/ID2:", len(train_keys & id2_keys))
print("Overlap ID/ID2:", len(id_keys & id2_keys))

Dropped 1081 duplicate rows (284807 -> 283726)

Split sizes and class balance:

Train: 100774 rows
{0: 100774}

ID: 21731 rows
{0: 21595, 1: 136}

ID2: 21731 rows
{0: 21595, 1: 136}

OOD: 139490 rows
{0: 139289, 1: 201}

Overlap Train/ID: 0
Overlap Train/ID2: 0
Overlap ID/ID2: 0


In [4]:
from sklearn.preprocessing import StandardScaler
import joblib

FEATURE_COLS = [c for c in df.columns if c not in ("Time", "Class")]
print(f"Model features ({len(FEATURE_COLS)}):", FEATURE_COLS)

scaler = StandardScaler()
scaler.fit(train_df[FEATURE_COLS])

def scale_split(d):
    out = d.copy()
    out[FEATURE_COLS] = scaler.transform(d[FEATURE_COLS])
    return out

train_scaled = scale_split(train_df)
id_scaled    = scale_split(id_df)
id2_scaled   = scale_split(id2_df)
ood_scaled   = scale_split(ood_df)

# Sanity: scaled Train features should be ~mean 0, std 1
print("\nTrain scaled feature means (should be ~0):")
print(train_scaled[FEATURE_COLS].mean().abs().max())
print("Train scaled feature stds (should be ~1):")
print(train_scaled[FEATURE_COLS].std().mean())

Model features (29): ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']

Train scaled feature means (should be ~0):
9.264821826947146e-17
Train scaled feature stds (should be ~1):
1.0000049616341646


In [5]:
SPLIT_DIR = os.path.join(OUTPUT_DIR, "splits", "credit_card")
os.makedirs(SPLIT_DIR, exist_ok=True)

train_scaled.to_parquet(os.path.join(SPLIT_DIR, "train.parquet"), index=False)
id_scaled.to_parquet(os.path.join(SPLIT_DIR, "id.parquet"), index=False)
id2_scaled.to_parquet(os.path.join(SPLIT_DIR, "id2.parquet"), index=False)
ood_scaled.to_parquet(os.path.join(SPLIT_DIR, "ood.parquet"), index=False)

joblib.dump(scaler, os.path.join(SPLIT_DIR, "scaler.joblib"))

config.update({
    "n_rows_raw": int(n_before),
    "n_duplicates_dropped": int(n_before - n_after),
    "n_rows_after_dedup": int(n_after),
    "temporal_cutoff_seconds": DAY_CUTOFF_SECONDS,
    "temporal_cutoff_rationale": "24h boundary (dataset spans ~48h); day 1 = train/ID/ID2 source, day 2 = OOD",
    "day1_benign_split_ratio": {"train": 0.70, "id": 0.15, "id2": 0.15},
    "day1_fraud_split_ratio": {"id": 0.5, "id2": 0.5, "train": 0.0},
    "model_features": FEATURE_COLS,
    "dropped_features": ["Time"],
    "dropped_features_rationale": "Time excluded as a model input to prevent the AE from trivially encoding ID-vs-OOD via raw elapsed-seconds rather than behavioral drift",
    "scaler": "StandardScaler, fit on Train (benign-only, Day 1) features only",
    "split_sizes": {
        "train": {"benign": int((train_df["Class"]==0).sum()), "fraud": int((train_df["Class"]==1).sum())},
        "id":    {"benign": int((id_df["Class"]==0).sum()),    "fraud": int((id_df["Class"]==1).sum())},
        "id2":   {"benign": int((id2_df["Class"]==0).sum()),   "fraud": int((id2_df["Class"]==1).sum())},
        "ood":   {"benign": int((ood_df["Class"]==0).sum()),   "fraud": int((ood_df["Class"]==1).sum())},
    },
    "seed": SEED,
})

with open(os.path.join(SPLIT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))
print(f"\nSaved to: {SPLIT_DIR}")
print(os.listdir(SPLIT_DIR))

{
  "notebook": "NB0a",
  "dataset": "creditcardfraud",
  "source": "mlg-ulb/creditcardfraud (Kaggle)",
  "seed": 42,
  "raw_path": "/kaggle/input/datasets/mlg-ulb/creditcardfraud/creditcard.csv",
  "n_rows_raw": 284807,
  "n_duplicates_dropped": 1081,
  "n_rows_after_dedup": 283726,
  "temporal_cutoff_seconds": 86400,
  "temporal_cutoff_rationale": "24h boundary (dataset spans ~48h); day 1 = train/ID/ID2 source, day 2 = OOD",
  "day1_benign_split_ratio": {
    "train": 0.7,
    "id": 0.15,
    "id2": 0.15
  },
  "day1_fraud_split_ratio": {
    "id": 0.5,
    "id2": 0.5,
    "train": 0.0
  },
  "model_features": [
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
    "V7",
    "V8",
    "V9",
    "V10",
    "V11",
    "V12",
    "V13",
    "V14",
    "V15",
    "V16",
    "V17",
    "V18",
    "V19",
    "V20",
    "V21",
    "V22",
    "V23",
    "V24",
    "V25",
    "V26",
    "V27",
    "V28",
    "Amount"
  ],
  "dropped_features": [
    "Time"
  ],
  "dropped_features_r

In [6]:
import os

CIC_DIR_HINTS = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith(".csv") and ("ISCX" in f or any(
            day in f for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
        )):
            full = os.path.join(root, f)
            size_mb = os.path.getsize(full) / (1024 * 1024)
            CIC_DIR_HINTS.append((full, round(size_mb, 1)))

if not CIC_DIR_HINTS:
    print("No CIC-IDS2017 files found under /kaggle/input.")
    print("Add a CIC-IDS2017 dataset via + Add Data (e.g. 'cicids2017' or 'network-intrusion-dataset').")
else:
    print(f"Found {len(CIC_DIR_HINTS)} file(s):\n")
    total_mb = 0
    for path, size_mb in sorted(CIC_DIR_HINTS):
        print(f"  {size_mb:>8.1f} MB  {path}")
        total_mb += size_mb
    print(f"\nTotal: {total_mb/1024:.2f} GB")

Found 8 file(s):

      73.6 MB  /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
      73.3 MB  /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
      55.6 MB  /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Morning.pcap_ISCX.csv
     168.7 MB  /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Monday-WorkingHours.pcap_ISCX.csv
      79.3 MB  /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
      49.6 MB  /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
     128.8 MB  /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Tuesday-WorkingHours.pcap_ISCX.csv
     214.7 MB  /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Wednesday-workingHours.pcap_ISCX.csv

Total: 0.82 GB


In [7]:
import os
import pandas as pd
import numpy as np

CIC_INPUT_DIR = "/kaggle/input/datasets/chethuhn/network-intrusion-dataset"
if not os.path.isdir(CIC_INPUT_DIR):
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(f.endswith("ISCX.csv") for f in files):
            CIC_INPUT_DIR = root
            break

FILES = sorted([f for f in os.listdir(CIC_INPUT_DIR) if f.lower().endswith(".csv")])
print(f"Loading {len(FILES)} files from {CIC_INPUT_DIR}")

def parse_day(fname):
    f = fname.lower()
    for day, order in [("monday",1),("tuesday",2),("wednesday",3),("thursday",4),("friday",5)]:
        if day in f:
            return day.capitalize(), order
    return "Unknown", -1

dfs = []
col_sets = {}
for fname in FILES:
    d = pd.read_csv(os.path.join(CIC_INPUT_DIR, fname), low_memory=False)
    d.columns = [c.strip() for c in d.columns]  # CICFlowMeter headers have leading spaces
    day, day_order = parse_day(fname)
    d["__source_file"] = fname
    d["__day"] = day
    d["__day_order"] = day_order
    col_sets[fname] = frozenset(d.columns) - {"__source_file", "__day", "__day_order"}
    print(f"  {fname}: {d.shape}")
    dfs.append(d)

base_cols = col_sets[FILES[0]]
print("\nColumn consistency check:")
all_match = True
for fname, cols in col_sets.items():
    if cols != base_cols:
        all_match = False
        print(f"  MISMATCH in {fname}: missing={base_cols-cols}, extra={cols-base_cols}")
print("All 8 files have identical columns:", all_match)

Loading 8 files from /kaggle/input/datasets/chethuhn/network-intrusion-dataset
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: (225745, 82)
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: (286467, 82)
  Friday-WorkingHours-Morning.pcap_ISCX.csv: (191033, 82)
  Monday-WorkingHours.pcap_ISCX.csv: (529918, 82)
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: (288602, 82)
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: (170366, 82)
  Tuesday-WorkingHours.pcap_ISCX.csv: (445909, 82)
  Wednesday-workingHours.pcap_ISCX.csv: (692703, 82)

Column consistency check:
All 8 files have identical columns: True


In [8]:
df = pd.concat(dfs, ignore_index=True)
print("Combined shape:", df.shape)
print(f"\nTotal columns: {len(df.columns)}")

print("\nOverall label distribution:")
print(df["Label"].value_counts())

print("\nLabel distribution by day (this determines the unseen-attack-type split):")
print(df.groupby("__day")["Label"].value_counts())

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
n_inf = np.isinf(df[numeric_cols]).sum().sum()
n_nan = df[numeric_cols].isna().sum().sum()
print(f"\nInf values across numeric cols: {n_inf}")
print(f"NaN values across numeric cols: {n_nan}")
if n_inf > 0:
    print("Columns with inf:", df[numeric_cols].columns[np.isinf(df[numeric_cols]).any()].tolist())
if n_nan > 0:
    print("Columns with NaN:", df[numeric_cols].columns[df[numeric_cols].isna().any()].tolist())

non_tag_cols = [c for c in df.columns if not c.startswith("__")]
print(f"\nDuplicate rows: {df.duplicated(subset=non_tag_cols).sum()}")

Combined shape: (2830743, 82)

Total columns: 82

Overall label distribution:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

Label distribution by day (this determines the unseen-attack-type split):
__day      Label                     
Friday     BENIGN                        414322
           PortScan                      158930
           DDoS                          128027
           Bot                             1966
Monday     BENI

In [9]:
# 1) Fix the mis-decoded labels (en-dash in "Web Attack – X" got corrupted on read)
label_fix = {
    "Web Attack \ufffd Brute Force": "Web Attack - Brute Force",
    "Web Attack \ufffd XSS": "Web Attack - XSS",
    "Web Attack \ufffd Sql Injection": "Web Attack - Sql Injection",
}
df["Label"] = df["Label"].replace(label_fix)
print("Labels after fix:", sorted(df["Label"].unique()))

# 2) Handle inf/NaN in Flow Bytes/s, Flow Packets/s (known divide-by-zero artifact
#    when Flow Duration == 0). These are a negligible fraction of rows — drop them
#    rather than impute, to avoid inventing flow-rate values for degenerate flows.
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "__day_order"]

n_before_clean = len(df)
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=numeric_cols).reset_index(drop=True)
n_after_clean = len(df)
print(f"\nDropped {n_before_clean - n_after_clean} rows with inf/NaN ({n_before_clean} -> {n_after_clean})")

# 3) Drop exact duplicate flows (well-documented artifact of this dataset),
#    keyed on the actual flow content — not on which file/day it came from.
feature_and_label_cols = [c for c in df.columns if not c.startswith("__")]
n_before_dedup = len(df)
df = df.drop_duplicates(subset=feature_and_label_cols).reset_index(drop=True)
n_after_dedup = len(df)
print(f"Dropped {n_before_dedup - n_after_dedup} duplicate rows ({n_before_dedup} -> {n_after_dedup})")

print(f"\nFinal shape: {df.shape}")
print("\nFinal label distribution by day:")
print(df.groupby("__day")["Label"].value_counts())

Labels after fix: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']

Dropped 2867 rows with inf/NaN (2830743 -> 2827876)
Dropped 307078 duplicate rows (2827876 -> 2520798)

Final shape: (2520798, 82)

Final label distribution by day:
__day      Label                     
Friday     BENIGN                        394777
           DDoS                          128014
           PortScan                       90694
           Bot                             1948
Monday     BENIGN                        502464
Thursday   BENIGN                        400256
           Web Attack - Brute Force        1470
           Web Attack - XSS                 652
           Infiltration                      36
           Web Attack - Sql Injection        21
Tuesday    BENIGN                        400833
        

In [10]:
print(f"Total columns: {len(df.columns)}\n")
for c in df.columns:
    print(repr(c))

# Flag columns that are identifiers/metadata rather than genuine flow features —
# these commonly leak identity/timing rather than reflecting traffic behavior
candidate_drop = [c for c in df.columns if any(
    kw in c.lower() for kw in ["flow id", "timestamp", "source ip", "destination ip", " ip"]
)]
print("\nCandidate columns to drop (identifiers/timestamp):", candidate_drop)

Total columns: 82

'Destination Port'
'Flow Duration'
'Total Fwd Packets'
'Total Backward Packets'
'Total Length of Fwd Packets'
'Total Length of Bwd Packets'
'Fwd Packet Length Max'
'Fwd Packet Length Min'
'Fwd Packet Length Mean'
'Fwd Packet Length Std'
'Bwd Packet Length Max'
'Bwd Packet Length Min'
'Bwd Packet Length Mean'
'Bwd Packet Length Std'
'Flow Bytes/s'
'Flow Packets/s'
'Flow IAT Mean'
'Flow IAT Std'
'Flow IAT Max'
'Flow IAT Min'
'Fwd IAT Total'
'Fwd IAT Mean'
'Fwd IAT Std'
'Fwd IAT Max'
'Fwd IAT Min'
'Bwd IAT Total'
'Bwd IAT Mean'
'Bwd IAT Std'
'Bwd IAT Max'
'Bwd IAT Min'
'Fwd PSH Flags'
'Bwd PSH Flags'
'Fwd URG Flags'
'Bwd URG Flags'
'Fwd Header Length'
'Bwd Header Length'
'Fwd Packets/s'
'Bwd Packets/s'
'Min Packet Length'
'Max Packet Length'
'Packet Length Mean'
'Packet Length Std'
'Packet Length Variance'
'FIN Flag Count'
'SYN Flag Count'
'RST Flag Count'
'PSH Flag Count'
'ACK Flag Count'
'URG Flag Count'
'CWE Flag Count'
'ECE Flag Count'
'Down/Up Ratio'
'Average Packe

In [11]:
# Fwd Header Length.1 is a documented CICFlowMeter export bug — identical to Fwd Header Length
assert (df["Fwd Header Length"] == df["Fwd Header Length.1"]).all(), "Columns differ — investigate before dropping"
df = df.drop(columns=["Fwd Header Length.1"])
print("Dropped duplicate 'Fwd Header Length.1' (confirmed identical to 'Fwd Header Length')")

META_COLS = ["__source_file", "__day", "__day_order", "Label"]
CANDIDATE_FEATURE_COLS = [c for c in df.columns if c not in META_COLS]
print(f"\nCandidate model features: {len(CANDIDATE_FEATURE_COLS)}")

Dropped duplicate 'Fwd Header Length.1' (confirmed identical to 'Fwd Header Length')

Candidate model features: 77


In [12]:
# Monday is 100% benign and shared by both Scenario T and Scenario A's training pool,
# so it's the right common reference for deciding which features carry no signal at all.
monday_benign = df[(df["__day"] == "Monday") & (df["Label"] == "BENIGN")]
variances = monday_benign[CANDIDATE_FEATURE_COLS].var()

zero_var_cols = variances[variances == 0].index.tolist()
near_zero_var_cols = variances[(variances > 0) & (variances < 1e-8)].index.tolist()

print(f"Zero-variance columns ({len(zero_var_cols)}):", zero_var_cols)
print(f"\nNear-zero-variance columns ({len(near_zero_var_cols)}):", near_zero_var_cols)

FEATURE_COLS = [c for c in CANDIDATE_FEATURE_COLS if c not in zero_var_cols]
print(f"\nFinal model feature count: {len(FEATURE_COLS)} (dropped {len(zero_var_cols)} zero-variance)")

Zero-variance columns (10): ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Near-zero-variance columns (0): []

Final model feature count: 67 (dropped 10 zero-variance)


In [13]:
df["Class"] = (df["Label"] != "BENIGN").astype(int)
print(df["Class"].value_counts())

Class
0    2095057
1     425741
Name: count, dtype: int64


In [14]:
TUE_ATTACKS_T = ["FTP-Patator", "SSH-Patator"]
WED_ATTACKS_T = ["DoS Hulk", "DoS GoldenEye", "DoS slowloris", "DoS Slowhttptest", "Heartbleed"]

benign_T = df[df["__day"].isin(["Monday", "Tuesday", "Wednesday"]) & (df["Class"] == 0)]
benign_T = benign_T.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

anomaly_T = df[
    ((df["__day"] == "Tuesday") & (df["Label"].isin(TUE_ATTACKS_T))) |
    ((df["__day"] == "Wednesday") & (df["Label"].isin(WED_ATTACKS_T)))
]
anomaly_T = anomaly_T.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_b = len(benign_T)
train_end = int(0.70 * n_b)
id_end = int(0.85 * n_b)
train_T = benign_T.iloc[:train_end].copy()
id_benign_T = benign_T.iloc[train_end:id_end].copy()
id2_benign_T = benign_T.iloc[id_end:].copy()

n_a = len(anomaly_T)
half = n_a // 2
id_anom_T = anomaly_T.iloc[:half].copy()
id2_anom_T = anomaly_T.iloc[half:].copy()

id_T = pd.concat([id_benign_T, id_anom_T], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
id2_T = pd.concat([id2_benign_T, id2_anom_T], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
ood_T = df[df["__day"].isin(["Thursday", "Friday"])].reset_index(drop=True)

print("=== Scenario T (temporal) ===")
for name, d in [("Train", train_T), ("ID", id_T), ("ID2", id2_T), ("OOD-temporal", ood_T)]:
    print(f"\n{name}: {len(d)} rows, Class balance: {d['Class'].value_counts().to_dict()}")

key_cols = FEATURE_COLS + ["Label", "__day"]
tr_keys, id_keys, id2_keys = (set(map(tuple, x[key_cols].values)) for x in (train_T, id_T, id2_T))
print("\nOverlap Train/ID:", len(tr_keys & id_keys))
print("Overlap Train/ID2:", len(tr_keys & id2_keys))
print("Overlap ID/ID2:", len(id_keys & id2_keys))

=== Scenario T (temporal) ===

Train: 910016 rows, Class balance: {0: 910016}

ID: 296457 rows, Class balance: {0: 195004, 1: 101453}

ID2: 296457 rows, Class balance: {0: 195004, 1: 101453}

OOD-temporal: 1017868 rows, Class balance: {0: 795033, 1: 222835}

Overlap Train/ID: 0
Overlap Train/ID2: 0
Overlap ID/ID2: 0


In [15]:
SEEN_ATTACKS_A = {
    "Tuesday": ["FTP-Patator", "SSH-Patator"],
    "Wednesday": ["DoS Hulk", "DoS GoldenEye", "DoS slowloris", "DoS Slowhttptest", "Heartbleed"],
    "Thursday": ["Web Attack - Brute Force", "Web Attack - XSS", "Infiltration", "Web Attack - Sql Injection"],
}

benign_A = df[df["__day"].isin(["Monday", "Tuesday", "Wednesday", "Thursday"]) & (df["Class"] == 0)]
benign_A = benign_A.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

anomaly_A = df[df.apply(lambda r: r["Label"] in SEEN_ATTACKS_A.get(r["__day"], []), axis=1)]
anomaly_A = anomaly_A.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_b = len(benign_A)
train_end = int(0.70 * n_b)
id_end = int(0.85 * n_b)
train_A = benign_A.iloc[:train_end].copy()
id_benign_A = benign_A.iloc[train_end:id_end].copy()
id2_benign_A = benign_A.iloc[id_end:].copy()

n_a = len(anomaly_A)
half = n_a // 2
id_anom_A = anomaly_A.iloc[:half].copy()
id2_anom_A = anomaly_A.iloc[half:].copy()

id_A = pd.concat([id_benign_A, id_anom_A], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
id2_A = pd.concat([id2_benign_A, id2_anom_A], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
ood_A = df[df["__day"] == "Friday"].reset_index(drop=True)

print("=== Scenario A (unseen-attack-type) ===")
for name, d in [("Train", train_A), ("ID", id_A), ("ID2", id2_A), ("OOD-attack", ood_A)]:
    print(f"\n{name}: {len(d)} rows, Class balance: {d['Class'].value_counts().to_dict()}")
print("\nOOD-attack Label breakdown (confirms these categories are genuinely unseen):")
print(ood_A[ood_A["Class"]==1]["Label"].value_counts())

tr_keys, id_keys, id2_keys = (set(map(tuple, x[key_cols].values)) for x in (train_A, id_A, id2_A))
print("\nOverlap Train/ID:", len(tr_keys & id_keys))
print("Overlap Train/ID2:", len(tr_keys & id2_keys))
print("Overlap ID/ID2:", len(id_keys & id2_keys))

# Confirm the core premise of Scenario A: none of Friday's attack categories appear in training
train_A_labels = set(train_A["Label"].unique()) | set(id_A["Label"].unique()) | set(id2_A["Label"].unique())
ood_A_attack_labels = set(ood_A[ood_A["Class"]==1]["Label"].unique())
print("\nFriday attack categories:", ood_A_attack_labels)
print("Any overlap with Train/ID/ID2 categories?:", train_A_labels & ood_A_attack_labels)

=== Scenario A (unseen-attack-type) ===

Train: 1190196 rows, Class balance: {0: 1190196}

ID: 357584 rows, Class balance: {0: 255042, 1: 102542}

ID2: 357585 rows, Class balance: {0: 255042, 1: 102543}

OOD-attack: 615433 rows, Class balance: {0: 394777, 1: 220656}

OOD-attack Label breakdown (confirms these categories are genuinely unseen):
Label
DDoS        128014
PortScan     90694
Bot           1948
Name: count, dtype: int64

Overlap Train/ID: 0
Overlap Train/ID2: 0
Overlap ID/ID2: 0

Friday attack categories: {'DDoS', 'PortScan', 'Bot'}
Any overlap with Train/ID/ID2 categories?: set()


In [16]:
from sklearn.preprocessing import StandardScaler
import joblib

def scale_scenario(train, id_, id2, ood, feature_cols, seed_note=""):
    scaler = StandardScaler()
    scaler.fit(train[feature_cols])
    def _scale(d):
        out = d.copy()
        out[feature_cols] = scaler.transform(d[feature_cols])
        return out
    return _scale(train), _scale(id_), _scale(id2), _scale(ood), scaler

train_T_s, id_T_s, id2_T_s, ood_T_s, scaler_T = scale_scenario(train_T, id_T, id2_T, ood_T, FEATURE_COLS)
train_A_s, id_A_s, id2_A_s, ood_A_s, scaler_A = scale_scenario(train_A, id_A, id2_A, ood_A, FEATURE_COLS)

# Sanity check both scalers
for name, tr in [("Scenario T", train_T_s), ("Scenario A", train_A_s)]:
    print(f"{name} — Train scaled: max|mean|={tr[FEATURE_COLS].mean().abs().max():.2e}, mean std={tr[FEATURE_COLS].std().mean():.4f}")

BASE_DIR = "/kaggle/working/splits/cic_ids2017"
for scenario, (tr, id_, id2, ood, scaler) in {
    "scenario_T": (train_T_s, id_T_s, id2_T_s, ood_T_s, scaler_T),
    "scenario_A": (train_A_s, id_A_s, id2_A_s, ood_A_s, scaler_A),
}.items():
    d = os.path.join(BASE_DIR, scenario)
    os.makedirs(d, exist_ok=True)
    tr.to_parquet(os.path.join(d, "train.parquet"), index=False)
    id_.to_parquet(os.path.join(d, "id.parquet"), index=False)
    id2.to_parquet(os.path.join(d, "id2.parquet"), index=False)
    ood.to_parquet(os.path.join(d, "ood.parquet"), index=False)
    joblib.dump(scaler, os.path.join(d, "scaler.joblib"))
    print(f"Saved {scenario} to {d}: {os.listdir(d)}")

cic_config = {
    "notebook": "NB0b/NB0c",
    "dataset": "CIC-IDS2017",
    "source": "chethuhn/network-intrusion-dataset (Kaggle)",
    "seed": SEED,
    "raw_files": FILES,
    "n_rows_raw": 2830743,
    "n_dropped_inf_nan": 2867,
    "n_dropped_duplicates": 307078,
    "n_rows_after_cleaning": len(df),
    "dropped_columns": ["Fwd Header Length.1"] + zero_var_cols,
    "dropped_columns_rationale": "Fwd Header Length.1 is a duplicate CICFlowMeter export bug; remaining 10 are zero-variance on Monday benign traffic",
    "model_features": FEATURE_COLS,
    "n_features": len(FEATURE_COLS),
    "scenarios": {
        "scenario_T": {
            "description": "Temporal shift: Mon+Tue+Wed (early) -> Thu+Fri (late)",
            "train_days": ["Monday", "Tuesday", "Wednesday"],
            "ood_days": ["Thursday", "Friday"],
            "id_id2_attack_categories": TUE_ATTACKS_T + WED_ATTACKS_T,
            "benign_split_ratio": {"train": 0.70, "id": 0.15, "id2": 0.15},
            "anomaly_split_ratio": {"id": 0.5, "id2": 0.5, "train": 0.0},
            "split_sizes": {k: {"benign": int((v["Class"]==0).sum()), "anomaly": int((v["Class"]==1).sum())}
                             for k, v in [("train",train_T),("id",id_T),("id2",id2_T),("ood",ood_T)]},
        },
        "scenario_A": {
            "description": "Unseen-attack-type shift: DDoS/PortScan/Bot (Friday) held out entirely from training",
            "train_days": ["Monday", "Tuesday", "Wednesday", "Thursday"],
            "ood_day": "Friday",
            "seen_attack_categories": SEEN_ATTACKS_A,
            "unseen_attack_categories": sorted(ood_A_attack_labels),
            "benign_split_ratio": {"train": 0.70, "id": 0.15, "id2": 0.15},
            "anomaly_split_ratio": {"id": 0.5, "id2": 0.5, "train": 0.0},
            "split_sizes": {k: {"benign": int((v["Class"]==0).sum()), "anomaly": int((v["Class"]==1).sum())}
                             for k, v in [("train",train_A),("id",id_A),("id2",id2_A),("ood",ood_A)]},
        },
    },
}

with open(os.path.join(BASE_DIR, "config.json"), "w") as f:
    json.dump(cic_config, f, indent=2, default=str)
print(f"\nConfig saved to {BASE_DIR}/config.json")

Scenario T — Train scaled: max|mean|=3.63e-16, mean std=1.0000
Scenario A — Train scaled: max|mean|=3.47e-16, mean std=1.0000
Saved scenario_T to /kaggle/working/splits/cic_ids2017/scenario_T: ['ood.parquet', 'id.parquet', 'scaler.joblib', 'train.parquet', 'id2.parquet']
Saved scenario_A to /kaggle/working/splits/cic_ids2017/scenario_A: ['ood.parquet', 'id.parquet', 'scaler.joblib', 'train.parquet', 'id2.parquet']

Config saved to /kaggle/working/splits/cic_ids2017/config.json


In [17]:
import shutil

shutil.make_archive("/kaggle/working/all_splits", "zip", "/kaggle/working/splits")
print("Created:", "/kaggle/working/all_splits.zip")
print("Size (MB):", os.path.getsize("/kaggle/working/all_splits.zip") / (1024*1024))

Created: /kaggle/working/all_splits.zip
Size (MB): 708.3357563018799
